In [33]:
import pandas as pd
import os
from ftfy import fix_text

In [34]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
# Load the "annotations" directory path from .env file
annotations_dir = os.getenv("ANNOTATIONS_DIR")
data_dir = os.getenv("DATA_DIR")


In [36]:
# List all .tsv files in the folder
files = os.listdir(annotations_dir)
for file in files:
	print(file)

california_wildfires_final_data.tsv
hurricane_harvey_final_data.tsv
hurricane_irma_final_data.tsv
hurricane_maria_final_data.tsv
iraq_iran_earthquake_final_data.tsv
mexico_earthquake_final_data.tsv
srilanka_floods_final_data.tsv


In [37]:
def map_to_general_disaster_type(specific_type):
	"""Map specific disaster types to more general categories"""
	if 'hurricane' in specific_type:
		return 'hurricane'
	elif 'wildfire' in specific_type:
		return 'wildfire'
	elif 'earthquake' in specific_type:
		return 'earthquake'
	elif 'flood' in specific_type:
		return 'flood'
	else:
		return 'other'

In [38]:
# Get all .tsv files in the "annotations" directory
tsv_files = [f for f in os.listdir(annotations_dir) if f.endswith(".tsv")]

# List to collect DataFrames for each disaster type
df_list = []

# Iterate over each .tsv file and read its data
print("Files read:")
for filename in tsv_files:
	# Add disaster type based on file name
	disaster_type = filename.replace('_final_data.tsv', '')

	file_path = os.path.join(annotations_dir, filename)
	df = pd.read_csv(file_path, sep="\t")
	print(f"{filename}: {len(df)} rows")

	# Add a column indicating the disaster type
	df["disaster_type"] = disaster_type

	# Add general disaster type
	df['general_disaster_type'] = df['disaster_type'].apply(map_to_general_disaster_type)

	df_list.append(df)

# Merge all DataFrames into a single DataFrame
annotations = pd.concat(df_list, ignore_index=True)
print(f"\nMerged file size: {len(annotations)} rows")

Files read:
california_wildfires_final_data.tsv: 1589 rows


hurricane_harvey_final_data.tsv: 4434 rows
hurricane_irma_final_data.tsv: 4504 rows
hurricane_maria_final_data.tsv: 4556 rows
iraq_iran_earthquake_final_data.tsv: 597 rows
mexico_earthquake_final_data.tsv: 1380 rows
srilanka_floods_final_data.tsv: 1022 rows

Merged file size: 18082 rows


In [39]:
annotations['tweet_text'] = annotations['tweet_text'].apply(fix_text)

In [40]:
pd.set_option('display.max_colwidth', None)
annotations['tweet_text'].head(5)

0          RT @Gizmodo: Wildfires raging through Northern California are terrifying https://t.co/dI73RFzX2i https://t.co/k4KnvIimsU
1                                       PHOTOS: Deadly wildfires rage in California https://t.co/td9xT3vXOL https://t.co/OimwAncLew
2    RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax
3    RT @Cal_OES: PLS SHARE: We're capturing wildfire response, recovery info here: https://t.co/r89LKpjLPj https://t.co/HiA1oQF2Ax
4          RT @TIME: California's raging wildfires as you've never seen them before https://t.co/OksQOZ2LHH https://t.co/oHTMbrM2Jx
Name: tweet_text, dtype: object

In [41]:
save_path = os.path.join(data_dir, "annotations.tsv")

# Save the merged DataFrame as a .tsv file
annotations.to_csv(save_path, sep='\t', index=False)

In [42]:
annotations.head(2)

,tweet_id,image_id,text_info,text_info_conf,image_info,image_info_conf,text_human,text_human_conf,image_human,image_human_conf,image_damage,image_damage_conf,tweet_text,image_url,image_path,disaster_type,general_disaster_type
0,917791044158185473,917791044158185473_0,informative,1.0,informative,0.6766,other_relevant_information,1.0,other_relevant_information,0.6766,NaN,NaN,RT @Gizmodo: Wildfires raging through Northern California are terrifying https://t.co/dI73RFzX2i https://t.co/k4KnvIimsU,http://pbs.twimg.com/media/DLyi_WYVYAApwNg.jpg,data_image/california_wildfires/10_10_2017/917791044158185473_0.jpg,california_wildfires,wildfire
1,917791130590183424,917791130590183424_0,informative,1.0,informative,0.6667,infrastructure_and_utility_damage,1.0,affected_individuals,0.6667,NaN,NaN,PHOTOS: Deadly wildfires rage in California https://t.co/td9xT3vXOL https://t.co/OimwAncLew,http://pbs.twimg.com/media/DLymKm9UMAAu0qw.jpg,data_image/california_wildfires/10_10_2017/917791130590183424_0.jpg,california_wildfires,wildfire


In [43]:
# Check if all the disaster types were read correctly
disasters_type = annotations["disaster_type"].unique()

print(disasters_type)
print(f"\n Total Natural Disasters: {str(len(disasters_type))}")

['california_wildfires' 'hurricane_harvey' 'hurricane_irma'
 'hurricane_maria' 'iraq_iran_earthquake' 'mexico_earthquake'
 'srilanka_floods']

 Total Natural Disasters: 7
